In [0]:
%run "/Workspace/TVAzureProject/Silver/Process_Silver_layer"

In [0]:
from pyspark.sql.functions import col, greatest, least, when, lit, sum as spark_sum, first
from delta.tables import DeltaTable

# ============================================
# Step 1: Join band x statement data with overlap condition
# ============================================
band_df = s_dim_30min_df
sd_df = s_sd_processed_df

cte_df = band_df.alias("band").join(
    sd_df.alias("sd"),
    (col("sd.event_date") == col("band.event_date")) &
    (col("sd.channel_id") == col("band.channel_id")) &
    (
        ((col("sd.start") >= col("band.start_sec")) & (col("sd.start") < col("band.end_sec"))) |
        ((col("sd.end") > col("band.start_sec")) & (col("sd.end") <= col("band.end_sec"))) |
        ((col("sd.start") < col("band.start_sec")) & (col("sd.end") > col("band.end_sec")))
    ),
    "inner"
)

# ============================================
# Step 2: Compute overlap / viewer_duration / timeband_duration (per session)
# ============================================
cte_df = cte_df.select(
    col("band.event_date").alias("event_date"),
    col("band.channel_id").alias("channel_id"),
    col("band.time_band").alias("timeband"),
    col("band.start_sec").alias("startseconds"),
    col("band.end_sec").alias("endseconds"),

    col("sd.member_id").alias("memberid"),
    col("sd.citygroup").alias("citygroup"),
    col("sd.gender").alias("gender"),
    col("sd.agegroup").alias("agegroup"),
    col("sd.actual_age").alias("actual_age"),
    col("sd.weight").alias("raw_weight"),
    col("sd.start").alias("viewer_startsec"),
    col("sd.end").alias("viewer_endsec"),

    (col("band.end_sec") - col("band.start_sec")).alias("timeband_duration"),

    when(
        (greatest(col("sd.start"), col("band.start_sec")) < least(col("sd.end"), col("band.end_sec"))) &
        (col("sd.start") > 0) & (col("sd.end") > 0),
        least(col("sd.end"), col("band.end_sec")) - greatest(col("sd.start"), col("band.start_sec"))
    ).otherwise(lit(0)).alias("viewer_duration")
)

cte_df = cte_df.withColumn("weight", col("raw_weight") / 1000.0).drop("raw_weight")

# ============================================
# Step 3: Aggregate multiple sessions per (event_date, channel_id, timeband, memberid)
# A member can have multiple separate viewing sessions within the same timeband —
# sum their durations rather than keeping just one (confirmed via viewer_startsec inspection)
# ============================================
agg_df = cte_df.groupBy(
    "event_date", "channel_id", "timeband", "memberid"
).agg(
    spark_sum("viewer_duration").alias("viewer_duration"),
    first("startseconds").alias("startseconds"),
    first("endseconds").alias("endseconds"),
    first("timeband_duration").alias("timeband_duration"),
    first("citygroup").alias("citygroup"),
    first("gender").alias("gender"),
    first("agegroup").alias("agegroup"),
    first("actual_age").alias("actual_age"),
    first("weight").alias("weight")
)

# Safety cap: viewer_duration should never exceed the timeband's total duration
agg_df = agg_df.withColumn(
    "viewer_duration",
    least(col("viewer_duration"), col("timeband_duration"))
)

# ============================================
# Step 4: AMA calculation on the aggregated (summed) duration
# ============================================
final_df = agg_df.withColumn(
    "amainth",
    when(col("timeband_duration") != 0,
         (col("viewer_duration") * col("weight")) / col("timeband_duration")
    ).otherwise(lit(None))
)

# ============================================
# Step 5: Null cleanup (weight/amainth -> 0)
# ============================================
final_df = final_df.withColumn(
    "weight", when(col("weight").isNull(), lit(0)).otherwise(col("weight"))
).withColumn(
    "amainth", when(col("amainth").isNull(), lit(0)).otherwise(col("amainth"))
)

# ============================================
# Step 6: Merge into gold table (incremental upsert)
# ============================================
target_table = "tv_cata.golden.g_tb_v_30m_ama_tvr"

if spark.catalog.tableExists(target_table):
    delta_table = DeltaTable.forName(spark, target_table)
    
    delta_table.alias("t").merge(
        final_df.alias("s"),
        """t.event_date = s.event_date 
           AND t.channel_id = s.channel_id 
           AND t.memberid = s.memberid 
           AND t.timeband = s.timeband"""
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    final_df.write.format("delta")\
        .mode("append")\
        .saveAsTable(target_table)

final_df.display()